In [ ]:
# Instalar ultralytics y sus dependencias desde cero
!pip install ultralytics
print("✅ Instalación completada")

✅ Instalación completada


In [ ]:
import cv2
import os
import shutil
import time
import numpy as np
from ultralytics import YOLO
from google.colab import drive
from google.colab import files

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Configuración de rutas
SRC_VIDEO = '/content/drive/MyDrive/Pickleball_Colab/partido1.mp4'
LOCAL_VIDEO = '/content/partido1_local.mp4'
DRIVE_PATH = '/content/drive/MyDrive/Pickleball_Colab'

# 3. Convertir video a formato local compatible
print("⚙️ Convirtiendo video a formato compatible...")
!ffmpeg -i {SRC_VIDEO} -c:v libx264 -preset fast -c:a copy -y {LOCAL_VIDEO}

if not os.path.exists(LOCAL_VIDEO):
    print("❌ Error al convertir el video.")
else:
    print("✅ Video convertido localmente.")

    cap = cv2.VideoCapture(LOCAL_VIDEO)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f" Video local: {width}x{height} @ {fps}fps")

    print("\n Cargando modelo YOLO (solo para jugadores)...")
    model_persons = YOLO("yolo11n.pt")
    print("✅ Modelo cargado")

    # 4. Configurar VideoWriter
    temp_video_path = '/content/temp_video.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(temp_video_path, fourcc, fps, (width, height))

    # 5. Procesar frames
    print(f"\n🚀 Procesando video con método Híbrido (Color + YOLO + Filtros)...")
    frame_num = 0
    start_time = time.time()
    max_frames = fps * 20

    kalman = None
    consecutive_misses = 0
    total_detections = 0

    # Rango HSV para Amarillo Fosforescente (confirmado en tu video)
    lower_yellow = np.array([20, 100, 100])
    upper_yellow = np.array([45, 255, 255])

    while True:
        ret, frame = cap.read()
        if not ret or frame_num >= max_frames:
            break

        persons_bboxes = []

        # --- A. Detectar jugadores con YOLO ---
        results_persons = model_persons(frame, classes=[0], verbose=False)
        for result in results_persons:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

                # Filtro para no marcar al árbitro o público
                if cy < height * 0.15 or cx < width * 0.08 or cx > width * 0.92:
                    continue

                persons_bboxes.append([int(x1), int(y1), int(x2), int(y2)])
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 255), 2)

        # --- B. Detectar pelota por COLOR ---
        ball_center = None
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, lower_yellow, upper_yellow)

        # Encontrar contornos
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        best_score = 0
        for cnt in contours:
            area = cv2.contourArea(cnt)

            # FILTRO 1: Tamaño (la pelota es pequeña pero no microscópica)
            if area < 15 or area > 600:
                continue

            # FILTRO 2: Forma (la pelota es casi redonda, aspect ratio cercano a 1)
            x, y, w, h = cv2.boundingRect(cnt)
            aspect_ratio = float(w) / h if h > 0 else 0
            if aspect_ratio < 0.4 or aspect_ratio > 2.5:
                continue  # Descartar líneas alargadas

            cx, cy = x + w//2, y + h//2

            # FILTRO 3: Ubicación (no dentro de un jugador)
            is_inside_person = False
            for px1, py1, px2, py2 in persons_bboxes:
                # Expandir un poco la caja del jugador para ser más estrictos
                margin = 10
                if (px1 - margin) < cx < (px2 + margin) and (py1 - margin) < cy < (py2 + margin):
                    is_inside_person = True
                    break

            if is_inside_person:
                continue

            # FILTRO 4: Zona válida (no en el logo ni en el graderío)
            if cx > width * 0.75 and cy > height * 0.75:
                continue
            if cy < height * 0.10:
                continue

            # Score: combinación de área y circularidad
            circularity = 4 * np.pi * area / (cv2.arcLength(cnt, True) ** 2) if cv2.arcLength(cnt, True) > 0 else 0
            score = area * circularity

            if score > best_score:
                best_score = score
                ball_center = (cx, cy)
                total_detections += 1

        # --- C. Tracking con Kalman ---
        if ball_center is None:
            consecutive_misses += 1
            if kalman is not None and consecutive_misses < 20:
                predicted = kalman.predict()
                ball_smooth = (int(predicted[0][0]), int(predicted[1][0]))
            else:
                ball_smooth = None
        else:
            consecutive_misses = 0
            x, y = ball_center
            measurement = np.array([[np.float32(x)], [np.float32(y)]])

            if kalman is None:
                kalman = cv2.KalmanFilter(4, 2)
                kalman.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
                kalman.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
                kalman.statePre = np.array([[np.float32(x)], [np.float32(y)], [0], [0]], np.float32)
                ball_smooth = (x, y)
            else:
                predicted = kalman.predict()
                pred_x, pred_y = int(predicted[0][0]), int(predicted[1][0])
                distance = np.sqrt((x - pred_x)**2 + (y - pred_y)**2)

                if distance > 200:
                    ball_smooth = (pred_x, pred_y)
                else:
                    kalman.correct(measurement)
                    corrected = kalman.statePost
                    ball_smooth = (int(corrected[0][0]), int(corrected[1][0]))

        # --- D. Dibujar pelota ---
        if ball_smooth:
            cv2.circle(frame, ball_smooth, 8, (0, 255, 0), -1)
            cv2.line(frame, (ball_smooth[0]-15, ball_smooth[1]),
                     (ball_smooth[0]+15, ball_smooth[1]), (0, 255, 0), 2)

        out.write(frame)
        frame_num += 1

        if frame_num % 50 == 0:
            elapsed = time.time() - start_time
            fps_real = 50 / elapsed if elapsed > 0 else 0
            print(f"  Frame {frame_num}/{max_frames} ({fps_real:.1f} fps reales) | Detecciones: {total_detections}")

    cap.release()
    out.release()
    print(f"\n✅ {frame_num} frames procesados.")
    print(f" Total de detecciones de pelota: {total_detections}")

    # 6. Convertir a H.264
    if frame_num > 0:
        output_local = '/content/partido_anotado.mp4'
        print("⏳ Convirtiendo a formato final...")
        !ffmpeg -i {temp_video_path} -c:v libx264 -pix_fmt yuv420p -preset fast -y {output_local}

        if os.path.exists(output_local):
            size_mb = os.path.getsize(output_local) / (1024 * 1024)
            print(f"\n🎉 Video creado: {size_mb:.2f} MB")
            drive_path = os.path.join(DRIVE_PATH, 'partido_anotado.mp4')
            shutil.copy(output_local, drive_path)
            print(f"💾 Guardado en Drive: {drive_path}")
            print("️ Iniciando descarga...")
            files.download(output_local)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚙️ Convirtiendo video a formato compatible...
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-li

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>